# Hands-on Fully Async DAPO / GRPO Training with verl on AMD Instinct™ GPUs

This tutorial is based on the following training script:

`verl/experimental/fully_async_policy/shell/dapo_7b_math_fsdp2_2_2.sh`

The core setup is:

- **Model**: `Qwen/Qwen2.5-Math-7B`
- **Training engine**: FSDP2
- **Rollout engine**: vLLM async mode
- **RL advantage estimator**: GRPO
- **Reward manager**: DAPO
- **Dataset**: DAPO-Math-17k
- **Validation**: AIME 2024
- **Hardware layout**: 4 GPUs total
  - 2 GPUs for training
  - 2 GPUs for rollout
- **Fully async settings**:
  - `staleness_threshold=0.1`
  - `trigger_parameter_sync_step=4`
  - `require_batches=4`
  - `partial_rollout=True`

The goal of this notebook is not to simply paste the shell script into one cell. Instead, we will break it down into:

1. Why Fully Async RL training is useful
2. How the verl Trainer and Rollouter are decoupled
3. What the DAPO / GRPO configuration controls
4. How 4 AMD GPUs are split into a 2 + 2 training / rollout layout
5. How to prepare the dataset and Qwen2.5-Math-7B
6. How to run preflight checks
7. How to launch, monitor, and stop training
8. How to understand staleness, parameter synchronization, and partial rollout

> **Important**
>
> The original script uses `trainer.save_freq=-1`, which means it does **not save checkpoints by default**.
> This tutorial preserves that behavior. At the end, an optional section explains what to change if you want to save checkpoints and run inference.

## Why Fully Async RL Training?

In a traditional colocated RL training loop, rollout and training often alternate:

```text
Generate rollouts
      ↓
Wait
      ↓
Train actor
      ↓
Wait
      ↓
Sync weights
      ↓
Generate next rollouts
```

For reasoning models, response lengths can vary significantly.

For example, within one prompt batch:

```text
sample A:  500 tokens
sample B: 1200 tokens
sample C: 3800 tokens
sample D: 4096 tokens
```

Even when shorter responses are already finished, the system may still need to wait for the longest response.

The main idea behind Fully Async training is to place the **Rollouter** and **Trainer** on different GPUs so they can run concurrently:

```text
                    ┌────────────────────┐
Prompts ───────────▶│  Rollouter / vLLM  │  GPU 2-3
                    └─────────┬──────────┘
                              │ generated samples
                              ▼
                       ┌─────────────┐
                       │Message Queue│
                       └──────┬──────┘
                              │
                              ▼
                    ┌────────────────────┐
                    │ Trainer / FSDP2    │  GPU 0-1
                    └─────────┬──────────┘
                              │
                              │ updated parameters
                              ▼
                    Parameter Synchronizer
                              │
                              └──────────────▶ Rollouter
```

This allows:

- The Rollouter to continuously generate new samples
- The Trainer to continuously consume already generated samples
- The two sides to avoid waiting for each other after every step
- Parameter synchronization to happen only after a configured number of training updates
- A limited amount of stale data to be tolerated to reduce pipeline bubbles

### Components Used in This Training Run

| Layer | Setting |
|---|---|
| Base model | `Qwen/Qwen2.5-Math-7B` |
| Training dataset | `DAPO-Math-17k` |
| Validation dataset | `AIME 2024` |
| Advantage estimator | `grpo` |
| Reward manager | `dapo` |
| Actor strategy | `FSDP2` |
| Rollout backend | `vLLM` |
| Rollout mode | `async` |
| Responses per prompt | `16` |
| Total GPUs | `4` |
| Trainer GPUs | `2` |
| Rollout GPUs | `2` |
| Tensor parallel for rollout | `1` |
| Actor FSDP size | `2` |
| Max prompt length | `2048` |
| Max response length | `4096` |
| Partial rollout | `True` |
| Staleness threshold | `0.1` |

## What Do verl and vLLM Do in Fully Async Training?

In this setup, verl does not use vLLM as just a normal inference library. Instead, vLLM acts as a dedicated rollout runtime.

```text
             Rollout side                             Training side

     ┌─────────────────────┐                 ┌─────────────────────────┐
     │ vLLM async servers  │                 │ FSDP2 Actor             │
     │                     │                 │                         │
     │ prompt              │                 │ consume trajectories    │
     │   ↓                 │                 │   ↓                     │
     │ generate n=16       │ ── samples ──▶  │ GRPO advantages         │
     │ responses           │                 │   ↓                     │
     └─────────────────────┘                 │ PPO-style clipped loss  │
                                             │   ↓                     │
                                             │ optimizer update        │
                                             └────────────┬────────────┘
                                                          │
                                                  parameter sync
                                                          │
                                                          ▼
                                                 vLLM gets new weights
```

The most important difference is:

**Rollout and training do not share the same GPUs.**

The script defines:

```bash
NGPUS_PER_NODE=4
n_gpus_rollout=2
n_gpus_training=2
```

So this is a **2 trainer + 2 rollout** resource-isolated setup.

## Step 1: Check AMD GPUs

Before launching verl, verify that the environment can see 4 AMD GPUs.

We want to confirm:

- ROCm / AMD GPU runtime is working
- At least 4 GPUs are visible
- GPU memory is not heavily occupied by other workloads

In [ ]:
!amd-smi

You can also verify the GPU count from PyTorch.

In [ ]:
import torch

hip_version = getattr(torch.version, "hip", None)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("PyTorch version :", torch.__version__)
print("HIP runtime     :", hip_version)
print("Visible GPUs    :", gpu_count)

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

assert hip_version is not None, "Current PyTorch is not a ROCm/HIP build."
assert gpu_count >= 4, f"This tutorial expects at least 4 GPUs, but only {gpu_count} are visible."

print("\nOK: ROCm + 4 GPU environment is ready.")

## Step 2: Verify verl and vLLM

The Fully Async Policy Trainer depends on:

- `verl`
- `vllm`
- PyTorch with ROCm support
- Ray

The next cell checks that all required components can be imported successfully.

In [ ]:
import verl
import vllm
import ray
import torch

print("verl :", getattr(verl, "__version__", "unknown"), verl.__file__)
print("vLLM :", vllm.__version__)
print("Ray  :", ray.__version__)
print("HIP  :", torch.version.hip)

print("\nEnvironment import check passed.")

## Step 3: Configure Paths Used by This Tutorial

The original shell script uses the following directory layout by default:

```text
${HOME}/verl/
├── models/
│   └── Qwen2.5-Math-7B/
├── data/
│   ├── dapo-math-17k.parquet
│   └── aime-2024.parquet
└── ckpts/
    └── DAPO/
```

The next cell defines these paths in one place.

If your verl repository is not located at `/workspace/verl`, update `VERL_DIR`.

In [ ]:
import os
from pathlib import Path

VERL_DIR = Path(os.environ.get("VERL_DIR", "/workspace/verl"))
RAY_DATA_HOME = Path(os.environ.get("RAY_DATA_HOME", str(Path.home() / "verl")))

MODEL_PATH = RAY_DATA_HOME / "models" / "Qwen2.5-Math-7B"
TRAIN_FILE = RAY_DATA_HOME / "data" / "dapo-math-17k.parquet"
TEST_FILE = RAY_DATA_HOME / "data" / "aime-2024.parquet"
os.environ["VERL_DIR"] = str(VERL_DIR)
os.environ["RAY_DATA_HOME"] = str(RAY_DATA_HOME)

CKPTS_DIR = (
    RAY_DATA_HOME
    / "ckpts"
    / "DAPO"
    / "DAPO-Qwen2.5-7b-MATH-0527a1-fsdp2-fully-async-2-2"
)

print("VERL_DIR      :", VERL_DIR)
print("RAY_DATA_HOME :", RAY_DATA_HOME)
print("MODEL_PATH    :", MODEL_PATH)
print("TRAIN_FILE    :", TRAIN_FILE)
print("TEST_FILE     :", TEST_FILE)
print("CKPTS_DIR     :", CKPTS_DIR)

## Step 4: Prepare DAPO-Math-17k and AIME 2024

The script expects two preprocessed parquet files in verl format:

```text
~/verl/data/dapo-math-17k.parquet
~/verl/data/aime-2024.parquet
```

The official DAPO recipe uses:

- Training: `BytedTsinghua-SIA/DAPO-Math-17k`
- Validation: `BytedTsinghua-SIA/AIME-2024`

The following cell will:

1. Create `~/verl/data`
2. Skip download if the files already exist
3. Download the prepared parquet files if they are missing

In [ ]:
%%bash
set -euo pipefail

RAY_DATA_HOME="${RAY_DATA_HOME:-${HOME}/verl}"
DATA_DIR="${RAY_DATA_HOME}/data"

TRAIN_FILE="${DATA_DIR}/dapo-math-17k.parquet"
TEST_FILE="${DATA_DIR}/aime-2024.parquet"

mkdir -p "${DATA_DIR}"

if [ -f "${TRAIN_FILE}" ]; then
  echo "Training data already exists: ${TRAIN_FILE}"
else
  echo "Downloading DAPO-Math-17k..."
  wget -O "${TRAIN_FILE}" \
    "https://huggingface.co/datasets/BytedTsinghua-SIA/DAPO-Math-17k/resolve/main/data/dapo-math-17k.parquet?download=true"
fi

if [ -f "${TEST_FILE}" ]; then
  echo "Validation data already exists: ${TEST_FILE}"
else
  echo "Downloading AIME-2024..."
  wget -O "${TEST_FILE}" \
    "https://huggingface.co/datasets/BytedTsinghua-SIA/AIME-2024/resolve/main/data/aime-2024.parquet?download=true"
fi

echo
ls -lh "${TRAIN_FILE}" "${TEST_FILE}"

### Inspect the Dataset

The DAPO dataset already contains the fields expected by the verl RL trainer, such as:

- `prompt`
- `reward_model`
- `data_source`
- `ability`
- `extra_info`

The training script uses:

```text
data.prompt_key=prompt
```

so the model reads the `prompt` field as input.

In [ ]:
import pandas as pd
from pathlib import Path

data_home = Path.home() / "verl" / "data"
train_file = data_home / "dapo-math-17k.parquet"
test_file = data_home / "aime-2024.parquet"

train_df = pd.read_parquet(train_file)
test_df = pd.read_parquet(test_file)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())

sample = train_df.iloc[0]

print("\n--- prompt ---")
print(sample["prompt"])

print("\n--- reward_model ---")
print(sample["reward_model"])

## Step 5: Prepare Qwen2.5-Math-7B

The original script expects the model at:

```text
~/verl/models/Qwen2.5-Math-7B
```

It also explicitly requires:

```json
"max_position_embeddings": 32768
```

to be set in `config.json`.

Even though this tutorial uses:

```text
max_prompt_length   = 2048
max_response_length = 4096
```

for a combined maximum of only `6144` tokens, we preserve the same model configuration as the original training recipe.

In [ ]:
%%bash
set -euo pipefail

RAY_DATA_HOME="${RAY_DATA_HOME:-${HOME}/verl}"
MODEL_PATH="${RAY_DATA_HOME}/models/Qwen2.5-Math-7B"

mkdir -p "$(dirname "${MODEL_PATH}")"

if [ -f "${MODEL_PATH}/config.json" ]; then
  echo "Model already exists at ${MODEL_PATH}, skipping download."
else
  echo "Downloading Qwen/Qwen2.5-Math-7B..."
  hf download Qwen/Qwen2.5-Math-7B --local-dir "${MODEL_PATH}"
fi

python3 - <<'PY'
import json
from pathlib import Path
import os

model_path = Path(os.environ.get(
    "MODEL_PATH",
    str(Path.home() / "verl" / "models" / "Qwen2.5-Math-7B")
))
config_path = model_path / "config.json"

with config_path.open() as f:
    config = json.load(f)

before = config.get("max_position_embeddings")
config["max_position_embeddings"] = 32768

with config_path.open("w") as f:
    json.dump(config, f, indent=2)
    f.write("\n")

print("config.json:", config_path)
print("max_position_embeddings:", before, "->", config["max_position_embeddings"])
PY

## Step 6: Understand What This Script Is Actually Training

### 6.1 GRPO Advantage Estimator

The script uses:

```text
algorithm.adv_estimator=grpo
actor_rollout_ref.rollout.n=16
```

For each prompt, the Rollouter generates **16 responses**:

```text
prompt
 ├── response 1
 ├── response 2
 ├── response 3
 ├── ...
 └── response 16
```

Each response receives its own reward.

The key idea behind GRPO is:

> Instead of training a separate critic to estimate value, GRPO compares rewards within the response group for the same prompt and computes relative advantages.

Therefore:

```text
n_resp_per_prompt = 16
```

is a key **group size** parameter in GRPO.

### 6.2 Is This “Full DAPO”?

This script uses:

```text
algorithm.adv_estimator=grpo
reward.reward_manager.name=dapo
clip_ratio_low=0.2
clip_ratio_high=0.28
clip_ratio_c=10.0
```

So it clearly includes several DAPO-style components:

- asymmetric / decoupled clipping
- DAPO reward manager
- overlong reward shaping
- token-level loss aggregation

However, the script does **not explicitly enable DAPO dynamic sampling / group filtering configuration**.

A more precise description is:

> **GRPO + Fully Async training using a DAPO-style recipe**

rather than claiming that every component from the DAPO paper is enabled.

### 6.3 Why Is KL Completely Disabled?

The script sets:

```text
algorithm.use_kl_in_reward=False
algorithm.kl_ctrl.kl_coef=0.0

actor_rollout_ref.actor.use_kl_loss=False
actor_rollout_ref.actor.kl_loss_coef=0.0
```

That means:

```text
No KL penalty in the reward
+
No KL loss in the actor objective
```

This differs from many PPO / GRPO recipes.

The update is mainly constrained by:

- the clipped policy objective
- the reward signal
- GRPO group-relative advantages

### 6.4 DAPO Clipping Configuration

The script uses:

| Config | Value |
|---|---:|
| `clip_ratio_low` | `0.20` |
| `clip_ratio_high` | `0.28` |
| `clip_ratio_c` | `10.0` |

This means clipping is not fully symmetric in both directions.

A standard PPO setup often uses something like:

```text
clip_ratio = 0.2
```

while this recipe separates it into:

```text
low  = 0.20
high = 0.28
```

which allows a larger update range in one direction.

### 6.5 Response Length and Overlong Buffer

The script sets:

```text
max_prompt_length   = 2048
max_response_length = 4096

overlong_buffer_len      = 4096
overlong_penalty_factor  = 1.0
```

and configures the reward manager as:

```text
reward.reward_manager.name=dapo
reward.reward_kwargs.overlong_buffer_cfg.enable=True
reward.reward_kwargs.overlong_buffer_cfg.len=4096
reward.reward_kwargs.overlong_buffer_cfg.penalty_factor=1.0
```

One purpose is to control a common behavior in reasoning models: generating excessively long responses.

If the model keeps extending the response purely for exploration, the reward function can penalize this behavior.

### 6.6 Memory Strategy for Actor / Reference / Rollout

| Component | Config | Meaning |
|---|---|---|
| Actor | `fsdp2` | The training model uses FSDP2 |
| Actor | `fsdp_size=2` | Actor parameters are sharded across 2 trainer GPUs |
| Actor param offload | `False` | Actor parameters stay on GPU |
| Actor optimizer offload | `False` | Optimizer states stay on GPU |
| Reference model | `param_offload=True` | Reference parameters may be offloaded to CPU |
| Rollout | `vllm` | vLLM generates responses |
| Rollout TP | `1` | No tensor parallelism inside each rollout engine |
| vLLM GPU util | `0.80` | vLLM may use a high fraction of rollout GPU memory |

### 6.7 Dynamic Batch Does Not Mean “Dynamically Changing Prompt Batch Size”

The script enables:

```text
actor_rollout_ref.actor.use_dynamic_bsz=True
actor_rollout_ref.ref.log_prob_use_dynamic_bsz=True
actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=True
```

Here, dynamic batching is mainly organized around a **token budget**.

Reasoning responses may have very different lengths:

```text
sample 1:  800 tokens
sample 2: 4000 tokens
```

If batches are built only by sample count, GPU workloads can become highly imbalanced.

The script therefore uses:

```text
actor_ppo_max_token_len_per_gpu
infer_ppo_max_token_len_per_gpu
```

to cap the number of tokens processed per GPU in a micro-batch.

In [ ]:
max_prompt_length = 2048
max_response_length = 4096

actor_ppo_max_token_len = (max_prompt_length + max_response_length) * 2
infer_ppo_max_token_len = (max_prompt_length + max_response_length) * 3
vllm_max_num_batched_tokens = max_prompt_length + max_response_length

print("prompt + response max length :", max_prompt_length + max_response_length)
print("actor token budget / GPU     :", actor_ppo_max_token_len)
print("ref/rollout token budget/GPU :", infer_ppo_max_token_len)
print("vLLM max batched tokens      :", vllm_max_num_batched_tokens)

## Step 7: Understand the Four Key Fully Async Parameters

This is the most important section of the tutorial.

The script uses:

```text
ppo_mini_batch_size         = 32
require_batches             = 4
trigger_parameter_sync_step = 4
staleness_threshold         = 0.1
partial_rollout             = True
```

### 7.1 `require_batches=4`

The Trainer does not update the model immediately after receiving a single rollout.

It waits for:

```text
require_batches × ppo_mini_batch_size
```

So:

```text
4 × 32 = 128
```

The Trainer collects **128 streaming samples** before performing one local training update.

Since each prompt generates:

```text
rollout.n = 16
```

each prompt corresponds to a 16-response GRPO group.

### 7.2 `trigger_parameter_sync_step=4`

The Trainer performs 4 local updates before synchronizing new parameters to the Rollouter.

Therefore, the number of samples processed between parameter synchronizations is:

```text
trigger_parameter_sync_step
× require_batches
× ppo_mini_batch_size
```

which is:

```text
4 × 4 × 32 = 512
```

This also makes the script's:

```text
total_rollout_steps = 512 × 100 = 51200
```

easy to interpret as roughly **100 logical 512-sample training intervals**.

In [ ]:
ppo_mini_batch_size = 32
require_batches = 4
trigger_parameter_sync_step = 4
rollout_n = 16
total_rollout_steps = 512 * 100

samples_per_local_update = require_batches * ppo_mini_batch_size
samples_between_sync = (
    trigger_parameter_sync_step
    * require_batches
    * ppo_mini_batch_size
)

print("Samples / local update      :", samples_per_local_update)
print("Local updates / param sync  :", trigger_parameter_sync_step)
print("Samples between param sync  :", samples_between_sync)
print("Responses per prompt        :", rollout_n)
print("Total rollout samples       :", total_rollout_steps)
print("Equivalent 512-sample units :", total_rollout_steps // samples_between_sync)

### 7.3 `staleness_threshold=0.1`

The tradeoff of Fully Async training is:

> The Trainer may be using a newer policy version than the one that generated some incoming rollout samples.

Those are **stale samples**.

```text
Rollouter uses policy v10
       ↓
generates sample
       ↓

Meanwhile:

Trainer updates:
v10 → v11 → v12

       ↓
sample generated by v10 reaches Trainer
```

`staleness_threshold=0.1` allows a limited amount of this stale data.

This keeps the pipeline moving while preventing stale off-policy samples from growing without control.

### 7.4 `partial_rollout=True`

Suppose a parameter synchronization is about to happen while vLLM is still generating a long response:

```text
generated:
token 1
token 2
...
token 1800
...
target max = 4096
```

Without partial rollout, the system may need to:

```text
wait for the response to finish
→ then synchronize parameters
```

With partial rollout enabled, the system can:

```text
pause an in-progress rollout
→ synchronize parameters
→ resume generation with newer parameters
```

This reduces waiting caused by long-tail responses.

In this script:

```text
staleness_threshold=0.1
partial_rollout=True
```

these two settings work together.

## Step 8: Run a Preflight Check

Before launching training, verify:

- The verl repository exists
- Training data exists
- AIME validation data exists
- The model exists
- `max_position_embeddings=32768`
- At least 4 GPUs are visible

In [ ]:
import json
from pathlib import Path
import torch
import os

verl_dir = Path(os.environ.get("VERL_DIR", "/workspace/verl"))
ray_data_home = Path(os.environ.get("RAY_DATA_HOME", str(Path.home() / "verl")))

model_path = ray_data_home / "models" / "Qwen2.5-Math-7B"
train_file = ray_data_home / "data" / "dapo-math-17k.parquet"
test_file = ray_data_home / "data" / "aime-2024.parquet"

checks = {
    "verl repo": verl_dir.exists(),
    "model": model_path.exists(),
    "train parquet": train_file.exists(),
    "validation parquet": test_file.exists(),
    "4 GPUs": torch.cuda.device_count() >= 4,
}

for name, ok in checks.items():
    print(f"{name:<22}: {'OK' if ok else 'MISSING'}")

assert all(checks.values()), "Preflight check failed."

with (model_path / "config.json").open() as f:
    config = json.load(f)

assert config.get("max_position_embeddings") == 32768, (
    "Please set max_position_embeddings=32768 in Qwen2.5-Math-7B/config.json"
)

print("\nAll preflight checks passed.")

## Step 9: Launch 2 + 2 Fully Async DAPO / GRPO Training

The next cell keeps the original shell script configuration nearly unchanged, with only two notebook-friendly adjustments:

1. It launches training with `nohup` so the notebook cell does not stay occupied
2. It redirects stdout / stderr to a dedicated `train_log.txt`

The training configuration remains:

```text
2 trainer GPUs + 2 rollout GPUs
FSDP2 + vLLM
GRPO + DAPO reward
Fully Async
```

> The original `trainer.save_freq=-1` is preserved, so checkpoints are not saved by default.

In [ ]:
%%bash
set -euo pipefail

VERL_DIR="${VERL_DIR:-/workspace/verl}"
RAY_DATA_HOME="${RAY_DATA_HOME:-${HOME}/verl}"

project_name="DAPO"
exp_name="DAPO-Qwen2.5-7b-MATH-0527a1-fsdp2-fully-async-2-2"

MODEL_PATH="${MODEL_PATH:-${RAY_DATA_HOME}/models/Qwen2.5-Math-7B}"
CKPTS_DIR="${CKPTS_DIR:-${RAY_DATA_HOME}/ckpts/${project_name}/${exp_name}}"
TRAIN_FILE="${TRAIN_FILE:-${RAY_DATA_HOME}/data/dapo-math-17k.parquet}"
TEST_FILE="${TEST_FILE:-${RAY_DATA_HOME}/data/aime-2024.parquet}"

rollout_mode="async"
rollout_name="vllm"

export VLLM_USE_V1=1
return_raw_chat="True"

# Algorithm
adv_estimator="grpo"

use_kl_in_reward=False
kl_coef=0.0
use_kl_loss=False
kl_loss_coef=0.0

clip_ratio_low=0.2
clip_ratio_high=0.28

# Length
max_prompt_length=$((1024 * 2))
max_response_length=$((1024 * 4))

enable_overlong_buffer=True
overlong_buffer_len=$((1024 * 4))
overlong_penalty_factor=1.0

loss_agg_mode="token-mean"

# Sampling
temperature=1.0
top_p=1.0
top_k=-1
val_top_p=0.7

# Performance
use_dynamic_bsz=True
actor_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 2))
infer_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 3))

ref_offload=True
actor_offload=False

gen_tp=1
sp_size=1
fsdp_size=2

# Fully async resource layout
NNODES="${NNODES:-1}"
NGPUS_PER_NODE="${NGPUS_PER_NODE:-4}"

n_gpus_rollout="${N_GPUS_ROLLOUT:-2}"
n_gpus_training=$((NGPUS_PER_NODE - n_gpus_rollout))

train_prompt_bsz=0
gen_prompt_bsz=1
n_resp_per_prompt=16
train_prompt_mini_bsz=32

total_rollout_steps=$((512 * 100))
test_freq=10

staleness_threshold=0.1
trigger_parameter_sync_step=4
require_batches=4
partial_rollout=True

TIMESTAMP=$(date +%Y%m%d.%H%M%S)
RUN_DIR="${RAY_DATA_HOME}/tutorial_runs/${exp_name}_${TIMESTAMP}"
mkdir -p "${RUN_DIR}"

echo "${RUN_DIR}" > /tmp/verl_fully_async_last_run

cd "${VERL_DIR}"

nohup python -m verl.experimental.fully_async_policy.fully_async_main \
    data.train_files="${TRAIN_FILE}" \
    data.val_files="${TEST_FILE}" \
    data.prompt_key=prompt \
    data.truncation='left' \
    data.max_prompt_length=${max_prompt_length} \
    data.max_response_length=${max_response_length} \
    data.train_batch_size=${train_prompt_bsz} \
    data.gen_batch_size=${gen_prompt_bsz} \
    data.return_raw_chat=${return_raw_chat} \
    actor_rollout_ref.rollout.n=${n_resp_per_prompt} \
    algorithm.adv_estimator=${adv_estimator} \
    algorithm.use_kl_in_reward=${use_kl_in_reward} \
    algorithm.kl_ctrl.kl_coef=${kl_coef} \
    actor_rollout_ref.actor.fsdp_config.strategy=fsdp2 \
    critic.strategy=fsdp2 \
    actor_rollout_ref.actor.use_kl_loss=${use_kl_loss} \
    actor_rollout_ref.actor.kl_loss_coef=${kl_loss_coef} \
    actor_rollout_ref.actor.clip_ratio_low=${clip_ratio_low} \
    actor_rollout_ref.actor.clip_ratio_high=${clip_ratio_high} \
    actor_rollout_ref.actor.clip_ratio_c=10.0 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.hybrid_engine=False \
    +actor_rollout_ref.model.override_config.max_position_embeddings=32768 \
    actor_rollout_ref.actor.use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.ref.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${actor_ppo_max_token_len} \
    actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.model.path="${MODEL_PATH}" \
    actor_rollout_ref.actor.optim.lr=1e-6 \
    actor_rollout_ref.actor.optim.lr_warmup_steps=10 \
    actor_rollout_ref.actor.optim.weight_decay=0.1 \
    actor_rollout_ref.actor.ppo_mini_batch_size=${train_prompt_mini_bsz} \
    actor_rollout_ref.actor.fsdp_config.param_offload=${actor_offload} \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=${actor_offload} \
    actor_rollout_ref.actor.entropy_coeff=0 \
    actor_rollout_ref.actor.grad_clip=1.0 \
    actor_rollout_ref.actor.loss_agg_mode=${loss_agg_mode} \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.80 \
    actor_rollout_ref.rollout.tensor_model_parallel_size=${gen_tp} \
    actor_rollout_ref.rollout.enable_chunked_prefill=True \
    actor_rollout_ref.rollout.max_num_batched_tokens=$((max_prompt_length + max_response_length)) \
    actor_rollout_ref.rollout.temperature=${temperature} \
    actor_rollout_ref.rollout.top_p=${top_p} \
    actor_rollout_ref.rollout.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.temperature=${temperature} \
    actor_rollout_ref.rollout.val_kwargs.top_p=${val_top_p} \
    actor_rollout_ref.rollout.val_kwargs.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.do_sample=True \
    actor_rollout_ref.rollout.val_kwargs.n=1 \
    actor_rollout_ref.rollout.calculate_log_probs=True \
    actor_rollout_ref.ref.fsdp_config.param_offload=${ref_offload} \
    actor_rollout_ref.ref.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.actor.fsdp_config.fsdp_size=${fsdp_size} \
    actor_rollout_ref.rollout.name=${rollout_name} \
    actor_rollout_ref.rollout.mode=${rollout_mode} \
    reward.reward_manager.name=dapo \
    +reward.reward_kwargs.overlong_buffer_cfg.enable=${enable_overlong_buffer} \
    +reward.reward_kwargs.overlong_buffer_cfg.len=${overlong_buffer_len} \
    +reward.reward_kwargs.overlong_buffer_cfg.penalty_factor=${overlong_penalty_factor} \
    +reward.reward_kwargs.overlong_buffer_cfg.log=False \
    +reward.reward_kwargs.max_resp_len=${max_response_length} \
    trainer.logger="['console','tensorboard']" \
    trainer.project_name="${project_name}" \
    trainer.experiment_name="${exp_name}" \
    trainer.val_before_train=True \
    trainer.save_freq=-1 \
    trainer.default_local_dir="${CKPTS_DIR}" \
    trainer.resume_mode=auto \
    trainer.nnodes="${NNODES}" \
    trainer.n_gpus_per_node="${n_gpus_training}" \
    rollout.nnodes="${NNODES}" \
    rollout.n_gpus_per_node="${n_gpus_rollout}" \
    rollout.total_rollout_steps="${total_rollout_steps}" \
    trainer.total_epochs=10 \
    trainer.test_freq="${test_freq}" \
    async_training.staleness_threshold="${staleness_threshold}" \
    async_training.trigger_parameter_sync_step="${trigger_parameter_sync_step}" \
    async_training.require_batches="${require_batches}" \
    async_training.partial_rollout="${partial_rollout}" \
    > "${RUN_DIR}/train_log.txt" 2>&1 &

PID=$!
echo "${PID}" > "${RUN_DIR}/train.pid"
disown

echo "Fully async training started."
echo "PID      : ${PID}"
echo "Run dir  : ${RUN_DIR}"
echo "Log file : ${RUN_DIR}/train_log.txt"
echo
echo "Resource split:"
echo "  trainer GPUs : ${n_gpus_training}"
echo "  rollout GPUs : ${n_gpus_rollout}"

### What Happens After Launch?

After training starts, the system does not run as:

```text
rollout → train → rollout → train
```

Instead, two long-running pipelines operate concurrently:

```text
GPU 0-1
Trainer:
consume → update → consume → update → ...

GPU 2-3
Rollouter:
generate → generate → generate → generate → ...
```

They are connected through:

```text
MessageQueue
+
ParameterSynchronizer
```

After every:

```text
4 local updates
```

the Trainer triggers a parameter synchronization.

## Step 10: Monitor the Training Log in Real Time

The next cell refreshes the latest log output every 3 seconds.

Stopping this cell **does not stop training**.

In Jupyter / JupyterLab, use Stop / Interrupt to stop watching the log.

In [ ]:
import time
import subprocess
from pathlib import Path
from IPython.display import clear_output

run_dir = Path(Path("/tmp/verl_fully_async_last_run").read_text().strip())
log_file = run_dir / "train_log.txt"
pid_file = run_dir / "train.pid"
pid = int(pid_file.read_text().strip())

def alive(process_id: int) -> bool:
    return Path(f"/proc/{process_id}").exists()

try:
    while True:
        result = subprocess.run(
            ["tail", "-n", "80", str(log_file)],
            capture_output=True,
            text=True,
        )

        clear_output(wait=True)
        status = "RUNNING" if alive(pid) else "EXITED"

        print(f"[{status}] PID={pid}")
        print(f"log={log_file}")
        print("-" * 100)
        print(result.stdout)

        if not alive(pid):
            break

        time.sleep(3)

except KeyboardInterrupt:
    print("\nStopped watching logs. Training continues in background.")

## Which Fully Async Metrics Should You Watch?

The biggest difference between Fully Async and standard colocated training is that, in addition to reward and loss, you should monitor **pipeline efficiency**.

Important metrics include:

| Metric | What to watch |
|---|---|
| `trainer/idle_ratio` | How much time the Trainer spends waiting for rollout data |
| `rollouter/idle_ratio` | How much time the Rollouter spends waiting for Trainer / sync |
| `fully_async/count/stale_samples_processed` | Number of stale samples processed |
| `fully_async/count/stale_trajectory_processed` | Number of stale trajectories |
| `fully_async/partial/total_partial_num` | Number of partial rollouts |
| `fully_async/partial/partial_ratio` | Fraction of rollouts that are partial |
| `fully_async/partial/max_partial_span` | How many policy versions a partial sample spans |

The goal is not to force any one metric to zero.

Instead, ask:

```text
Is trainer idle time too high?
Is rollouter idle time too high?
Are stale samples growing continuously?
Is partial rollout reducing synchronization stalls?
Are reward and validation metrics still stable?
```

### How Should You Adjust the 2 + 2 GPU Split Based on Idle Ratio?

The current script uses:

```text
Trainer  : 2 GPUs
Rollouter: 2 GPUs
```

If you observe:

```text
trainer/idle_ratio is high
rollouter/idle_ratio is low
```

that suggests:

> The Trainer is frequently starved for data, so rollout generation is relatively slow.

In that case, consider allocating more resources to rollout.

On the other hand:

```text
rollouter/idle_ratio is high
trainer/idle_ratio is low
```

suggests:

> Rollout has spare capacity, while the Trainer is slower at consuming and updating.

In that case, consider allocating more training resources.

This is the value of Fully Async resource isolation: the Trainer and Rollouter GPU counts can be tuned independently.

## TensorBoard

The script uses:

```text
trainer.logger=['console','tensorboard']
```

so you can also inspect training curves with TensorBoard.

If event files are written under the experiment directories, you can launch:

```bash
tensorboard --logdir ~/verl --port 6006 --bind_all
```

and then open the corresponding port in your browser.

In [ ]:
%%bash
echo "Search TensorBoard event files:"
find "${HOME}/verl" -type f -name "events.out.tfevents.*" 2>/dev/null | tail -20

## How to Stop the Current Training Run

Fully Async training launches multiple processes, including Ray workers and vLLM servers.

Therefore, killing only the Python launcher PID may not immediately release all GPU resources.

The next cell will:

1. Send `TERM` to the launcher
2. Send `KILL` if needed
3. Leave instructions for inspecting remaining Fully Async / Ray / vLLM processes

> If the machine is shared with other verl / Ray / vLLM workloads, do not blindly kill all matching processes.

In [ ]:
%%bash
set -euo pipefail

RUN_DIR=$(cat /tmp/verl_fully_async_last_run)
PID=$(cat "${RUN_DIR}/train.pid")

echo "Stopping PID=${PID}"
kill -TERM "${PID}" 2>/dev/null || true

for _ in 1 2 3 4 5; do
  [ ! -d "/proc/${PID}" ] && break
  sleep 1
done

if [ -d "/proc/${PID}" ]; then
  kill -KILL "${PID}" 2>/dev/null || true
fi

echo
echo "Launcher stop request completed."
echo
echo "If GPU memory is still occupied, inspect processes first:"
echo "  pgrep -af 'fully_async_main|ray::|vllm'"
echo
echo "Do NOT blindly pkill shared Ray/vLLM processes on a multi-user machine."

## Optional: Save Checkpoints and Run Inference

The original script uses:

```text
trainer.save_freq=-1
```

which means:

> **No checkpoints are saved.**

If your goal is only:

- Fully Async performance validation
- training convergence / AIME validation
- throughput comparison
- idle ratio / stale sample analysis

then keeping `-1` is fine.

But if you want to load the trained actor for inference, change:

```text
trainer.save_freq=-1
```

to something like:

```text
trainer.save_freq=10
```

and restart training.

### FSDP2 Checkpoint Merge

Once checkpoints are available, you can typically merge actor FSDP shards into Hugging Face format:

```bash
python -m verl.model_merger merge \
  --backend fsdp \
  --local_dir <checkpoint>/actor \
  --target_dir <checkpoint>/actor/merged_hf
```

Then load the merged model with:

```python
AutoTokenizer.from_pretrained(...)
AutoModelForCausalLM.from_pretrained(...)
```

This step is intentionally excluded from the default execution path because it would change the behavior of the original `save_freq=-1` script.

## Putting the Entire Training Flow Together

```text
DAPO-Math-17k prompts
        │
        ▼
┌─────────────────────────┐
│ Rollouter: vLLM async   │
│ GPUs: 2                 │
│ n = 16                  │
│ temp = 1.0              │
│ top_p = 1.0             │
└───────────┬─────────────┘
            │
            │ streaming trajectories
            ▼
      ┌──────────────┐
      │ MessageQueue │
      └───────┬──────┘
              │
              │ 4 × 32 = 128 samples
              ▼
┌─────────────────────────┐
│ Trainer: FSDP2          │
│ GPUs: 2                 │
│ GRPO advantage          │
│ DAPO reward             │
│ clip: 0.20 / 0.28       │
└───────────┬─────────────┘
            │
            │ 4 local updates
            ▼
┌─────────────────────────┐
│ Parameter Synchronizer  │
└───────────┬─────────────┘
            │
            └──────────────▶ Rollouter gets newer weights

staleness_threshold = 0.1
partial_rollout = True
```

In one sentence:

> **This setup continuously trains on 2 GPUs and continuously generates rollouts on 2 GPUs, using a streaming queue, bounded staleness, and partial rollout to overlap generation with actor updates and reduce GPU idle time caused by long-tail reasoning responses and frequent synchronization.**

## Three Experiments to Try After This Tutorial

### Experiment 1 — Fully Async vs Colocate

Keep the following as consistent as possible:

- model
- dataset
- rollout.n
- sequence length
- total rollout samples

Then compare:

```text
wall-clock time
trainer idle ratio
rollouter idle ratio
AIME validation
```

### Experiment 2 — Tune Staleness

Try:

```text
0
0.1
0.5
```

and observe:

```text
throughput
stale samples
validation accuracy
response length
```

### Experiment 3 — Tune Trainer / Rollouter GPU Ratio

On a machine with more GPUs, try:

```text
2 trainer + 2 rollout
4 trainer + 4 rollout
2 trainer + 6 rollout
6 trainer + 2 rollout
```

Use:

```text
trainer/idle_ratio
rollouter/idle_ratio
```

to determine whether the bottleneck is generation or training.

## References

- Training script used by this tutorial  
  `https://github.com/Vivicai1005/verl/blob/rocm/verl/experimental/fully_async_policy/shell/dapo_7b_math_fsdp2_2_2.sh`

- Reference workshop notebook  
  `https://github.com/Vivicai1005/rl-workshop/blob/main/verl_qwen3_4b_lora_grpo_workshop.ipynb`

- verl Fully Async Policy Trainer  
  `https://github.com/verl-project/verl/blob/main/docs/advance/fully_async.md`

- DAPO dataset  
  `https://huggingface.co/datasets/BytedTsinghua-SIA/DAPO-Math-17k`

- AIME 2024 validation dataset  
  `https://huggingface.co/datasets/BytedTsinghua-SIA/AIME-2024`